# PM2.5 零樣本預測：完整實驗（Colab Enterprise）

1. 連線到 GPU 執行階段後，把 `pm25_project.zip` 上傳到左側 📁 檔案瀏覽器。
2. 按「全部執行」。**任何一格出錯都會停下並顯示紅色錯誤**，請截圖給 Claude。
3. 實驗可重跑續跑：已完成的「測站 × 模型」會自動跳過。
4. 最後一格會下載 `pm25_results.zip`。
5. **跑完務必停止執行階段**，否則會持續扣額度。

In [ ]:
%%bash
set -e
# 1. 檢查 GPU 與 Python 版本
nvidia-smi || echo '沒有 GPU，將使用 CPU（較慢但可跑）'
python --version

In [ ]:
# 2. 解壓縮專案（重新上傳新版 zip 時會覆蓋舊程式，但保留已完成的預測）
import os
os.chdir('/content' if os.path.isdir('/content') else os.getcwd())
assert os.path.exists('pm25_project.zip'), '找不到 pm25_project.zip，請先上傳到左側檔案瀏覽器'
!rm -rf pm25/src pm25/data/processed
!unzip -q -o pm25_project.zip -d pm25
os.chdir('pm25')
print(os.getcwd())
!ls data/processed | head -3

In [ ]:
%%bash
set -e
# 3. 安裝套件（numba 的 numpy 版本警告可忽略）
pip install -q -e '.[models]'
python -c "import pandas, torch; from pm25.models.device import default_device; print('pandas', pandas.__version__, '| torch', torch.__version__, '| device =', default_device())"

## 實驗
Chronos／LSTM（torch）與 XGBoost 必須在不同行程執行，以下每一格都是獨立行程。

In [ ]:
%%bash
set -e
# 4. Persistence 基準（幾秒）
python -m pm25.experiment --family baseline

In [ ]:
%%bash
set -e
# 5. Chronos-Bolt Small/Base、Chronos-2（單變量／共變數／未來風場 oracle）
# L4 上預估 10–30 分鐘；第一次會下載模型權重
python -m pm25.experiment --family chronos

In [ ]:
%%bash
set -e
# 6. XGBoost（CPU）
python -m pm25.experiment --family xgboost

In [ ]:
%%bash
set -e
# 7. LSTM（GPU）
python -m pm25.experiment --family lstm

## 彙整與下載

In [ ]:
# 8. 確認預測檔齊全後產生指標表
import glob
files = glob.glob('results/predictions/*.csv.gz')
print('預測檔數量：', len(files), '/ 96')
assert len(files) == 96, '預測檔不齊全，請往上檢查哪一格出錯'
!python -m pm25.evaluation.report

In [ ]:
# 9. 打包結果並下載（也可在左側檔案瀏覽器 /content 對 pm25_results.zip 按右鍵下載）
!cd .. && rm -f pm25_results.zip && zip -q -r pm25_results.zip pm25/results
!ls -lh ../pm25_results.zip
from google.colab import files
files.download('../pm25_results.zip')

## 結束
到 Colab Enterprise →「執行階段」把這個執行階段**停止或刪除**，避免繼續計費。